In [1]:
import sys
!{sys.executable} -m pip install fredapi yfinance alpha_vantage


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip


# Scraping Data

## FredAPI (MUST GET YOUR OWN API KEY)

In [2]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key='b29fa8db713c7d9498fc161a88fb77ac')

#UNRATE: unemployment rate
#CPIAUCSL: consumer price index (inflaton)
#FEDFUNDS: effective federal funds rate (interest rates)
#GDPC1: Real Gross Domestic Product
#T10Y2Y: Yield Curve (recession indicator)
#TEDRATE: TED Spread (interbank credit risk)
#VIXLS: CBOE (market fear)
#BAMLH0A0HYM2: Bank of America's US High Yield Index (Credit risk)

indexes = ['UNRATE', 'CPIAUCSL', 'FEDFUNDS', 'GDPC1', 'T10Y2Y', 'TEDRATE', 'VIXCLS', 'BAMLH0A0HYM2']

data = {}
for index in indexes:
    data[index] = fred.get_series(index)

df_macro = pd.DataFrame(data)
df_macro.index.name = 'Date'
df_macro = df_macro.loc['2000-01-01':]

In [3]:
df_macro.head()

,UNRATE,CPIAUCSL,FEDFUNDS,GDPC1,T10Y2Y,TEDRATE,VIXCLS,BAMLH0A0HYM2
Date,,,,,,,,
2000-01-01,4.0,169.3,5.45,13878.147,NaN,NaN,NaN,NaN
2000-01-03,NaN,NaN,NaN,NaN,0.20,NaN,24.21,4.68
2000-01-04,NaN,NaN,NaN,NaN,0.19,0.77,27.01,4.81
2000-01-05,NaN,NaN,NaN,NaN,0.24,0.75,26.41,4.77
2000-01-06,NaN,NaN,NaN,NaN,0.22,0.78,25.73,4.82


## Yahoo Finance API

In [4]:
import yfinance as yf

#ticker symbols
# CL=F: crude oil futures
# GC=F: gold futures 
# XLE: energy select sector (correlation with oil)
# XLK: technology select sector
# XLF: financial sector (rate sensistive)
# GSPC: S and P 500 Index
# TNX: 10 year treasuring yield

tickers = ['CL=F', 'GC=F', 'XLE', 'XLK', 'XLF', '^GSPC', '^TNX']
df_sector = yf.download(tickers, start="2000-01-01", interval="1d", auto_adjust=True)['Close']

[*********************100%***********************]  7 of 7 completed


In [5]:
df_sector.head()

Ticker,CL=F,GC=F,XLE,XLF,XLK,^GSPC,^TNX
Date,,,,,,,
2000-01-03,NaN,NaN,6.699404,11.147914,20.652037,1455.219971,6.548
2000-01-04,NaN,NaN,6.573301,10.660571,19.604298,1399.420044,6.485
2000-01-05,NaN,NaN,6.746696,10.576809,19.313257,1402.109985,6.599
2000-01-06,NaN,NaN,7.006789,11.041310,18.672968,1403.449951,6.549
2000-01-07,NaN,NaN,7.081662,11.224062,18.998934,1441.469971,6.504


In [6]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [7]:
df_main = df_sector.join(df_macro, how='outer')#.join(df_fx, how='outer')
df_main.sort_index(inplace=True)

# forward fill bc market uses that monthly or weekly number
df_main.fillna(method='ffill', inplace=True)
df_main.dropna(inplace=True)
df_main

/var/folders/n9/hrwgbvx936vf2n8y560vb_lc0000gn/T/ipykernel_16180/275107353.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_main.fillna(method='ffill', inplace=True)


,CL=F,GC=F,XLE,XLF,XLK,^GSPC,^TNX,UNRATE,CPIAUCSL,FEDFUNDS,GDPC1,T10Y2Y,TEDRATE,VIXCLS,BAMLH0A0HYM2
Date,,,,,,,,,,,,,,,
2000-08-30,33.400002,273.899994,8.264154,13.620219,20.652037,1502.589966,5.800,4.1,172.700,6.50,14145.312,-0.43,0.54,17.69,6.28
2000-08-31,33.099998,278.299988,8.232369,13.972992,21.059486,1517.680054,5.729,4.1,172.700,6.50,14145.312,-0.45,0.55,16.84,6.43
2000-09-01,33.380001,277.000000,8.315806,13.742926,21.071123,1520.770020,5.675,3.9,173.600,6.52,14145.312,-0.41,0.58,17.53,6.48
2000-09-04,33.380001,277.000000,8.315806,13.742926,21.071123,1520.770020,5.675,3.9,173.600,6.52,14145.312,-0.41,0.58,17.53,6.49
2000-09-05,33.799999,275.799988,8.315806,13.911640,20.593821,1507.079956,5.683,3.9,173.600,6.52,14145.312,-0.40,0.58,19.82,6.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-03,74.559998,5107.399902,56.520000,51.209999,137.500000,6816.629883,4.056,4.4,326.588,3.64,24111.830,0.55,0.09,23.57,3.08
2026-03-04,74.660004,5120.200195,56.189999,51.500000,139.839996,6869.500000,4.080,4.4,326.588,3.64,24111.830,0.55,0.09,21.15,2.97
2026-03-05,81.010002,5065.299805,56.480000,51.230000,140.179993,6830.709961,4.146,4.4,326.588,3.64,24111.830,0.56,0.09,23.75,3.00


In [8]:
df_main.drop(columns=['GDPC1'], inplace=True)
df_main.drop(columns=['TEDRATE'], inplace=True)

In [9]:
df_main.to_csv('Financial_Data.csv')

In [10]:
df_main = pd.read_csv("Financial_Data.csv")
df_index = df_main['Date']
df_main.drop(columns=['Date'], inplace=True)
df_main = df_main.astype(float)
df_main.index = df_index
df_main

,CL=F,GC=F,XLE,XLF,XLK,^GSPC,^TNX,UNRATE,CPIAUCSL,FEDFUNDS,T10Y2Y,VIXCLS,BAMLH0A0HYM2
Date,,,,,,,,,,,,,
2000-08-30,33.400002,273.899994,8.264154,13.620219,20.652037,1502.589966,5.800,4.1,172.700,6.50,-0.43,17.69,6.28
2000-08-31,33.099998,278.299988,8.232369,13.972992,21.059486,1517.680054,5.729,4.1,172.700,6.50,-0.45,16.84,6.43
2000-09-01,33.380001,277.000000,8.315806,13.742926,21.071123,1520.770020,5.675,3.9,173.600,6.52,-0.41,17.53,6.48
2000-09-04,33.380001,277.000000,8.315806,13.742926,21.071123,1520.770020,5.675,3.9,173.600,6.52,-0.41,17.53,6.49
2000-09-05,33.799999,275.799988,8.315806,13.911640,20.593821,1507.079956,5.683,3.9,173.600,6.52,-0.40,19.82,6.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-03,74.559998,5107.399902,56.520000,51.209999,137.500000,6816.629883,4.056,4.4,326.588,3.64,0.55,23.57,3.08
2026-03-04,74.660004,5120.200195,56.189999,51.500000,139.839996,6869.500000,4.080,4.4,326.588,3.64,0.55,21.15,2.97
2026-03-05,81.010002,5065.299805,56.480000,51.230000,140.179993,6830.709961,4.146,4.4,326.588,3.64,0.56,23.75,3.00
